### Notebook to develop and test the implementation of MTEB for sparse vectorization

In [1]:
import re 
import mteb
from src import tfidf_for_mteb
#from threadpoolctl import threadpool_limits

# autoreload to keep track of changes
#%load_ext autoreload
#%autoreload 1
#%aimport src.tfidf_for_mteb 

#### Step 1: write a function to get a vocabulary list from a list of strings

In [ ]:
# using regex to find words for the vocabulary list 
# This function needs to be callable by the evaluators in sparse_mteb/mteb/evaluation
# therefore, it is copied into sparse_mteb/mteb/evaluation/evaluators/utils.py

def get_vocab(text: [str], token_pattern: str = r"(?u)\b\w\w+\b", lowercase: bool = True) -> [str]:
    """return a list of unique words ocurring in text that fulfill the specified token_pattern
    The default token_pattern is the one used in the scikit-learn TfidfVectorizer class
    """
    if lowercase:
        vocab = [word for sent in text for word in re.findall(token_pattern, sent.lower())]
    else:
        vocab = [word for sent in text for word in re.findall(token_pattern, sent)]
    
    return list(set(vocab))


In [ ]:
test_list = ["These are some words. What else?", "I've thought hard about what words to write.", "Words won't be enough!"]
get_vocab(text=test_list)

#### Step 2: Try mteb tasks with tfidf

In [2]:
# define models to compare
tfidf_model = tfidf_for_mteb.Tfidf()
glove_model = mteb.get_model("sentence-transformers/average_word_embeddings_glove.6B.300d")

##### Bitext Mining 

Task: translation: match sentence from set 1 to sentence from set 2 (in different language)

Implementation for TF-IDF: tfidf is obviously horrible at this. the sets could either be embedded together (which is how I implemented it for now), which means that the translation for each word would just be another matrix entry, or embedded separately, which also makes no sense because the first word in the english vocab will not align with the first word in french 

Changes implemented in which files?

In [ ]:
tasks = mteb.get_tasks(tasks=["FloresBitextMining"])
evaluation = mteb.MTEB(tasks=tasks)
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")


In [ ]:
glove_results = evaluation.run(glove_model, output_folder="../MTEB/sparse_results")

##### Classification

**Task**: train logistic regression classifier on train set embeddings, evaluate on performance for test set embeddings

**Implementation for TF-IDF:** compute vocab for embeddings for the union of train and test set. For multilingual datasets, the embeddings are computed for each language separately to reduce the size of embeddings (this shouldn't lead to conflicts because each language is evaluated separately)

**Changes implemented in which files?**
- AbsTaskClassification.py

**Problem Tasks**
- AmazonPolarityClassification
- probably also AmazonReviewsClassification

In [3]:
tasks = mteb.get_tasks(tasks=["MassiveIntentClassification"])
evaluation = mteb.MTEB(tasks=tasks)

In [4]:
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")

───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- MassiveIntentClassification, s2s, multilingual 51 / 51 Subsets

README.md:   0%|          | 0.00/9.25k [00:00<?, ?B/s]

da.json.gz:   0%|          | 0.00/192k [00:00<?, ?B/s]

...

zh-CN.json.gz:   0%|          | 0.00/42.4k [00:00<?, ?B/s]

th.json.gz:   0%|          | 0.00/49.3k [00:00<?, ?B/s]

ta.json.gz:   0%|          | 0.00/52.4k [00:00<?, ?B/s]

zh-TW.json.gz:   0%|          | 0.00/40.7k [00:00<?, ?B/s]

dataset keys (called in abstaskclassification) dict_keys(['am', 'sq', 'ko', 'cy', 'ta', 'pl', 'ka', 'ms', 'bn', 'is', 'en', 'az', 'nb', 'fi', 'id', 'lv', 'ml', 'kn', 'sl', 'jv', 'hy', 'my', 'es', 'zh-TW', 'sv', 'el', 'mn', 'pt', 'te', 'hi', 'sw', 'ar', 'fa', 'da', 'fr', 'he', 'ja', 'de', 'vi', 'ur', 'tl', 'ru', 'tr', 'hu', 'km', 'nl', 'it', 'ro', 'th', 'af', 'zh-CN'])
dataset keys: dict_keys(['train', 'test', 'validation'])
get_vocab called!
input: <class 'list'> of length 16521
vocab of lenght 13209 starting with ['ትይዝልኝ', 'ክፍተት', 'ፍጡራንን', 'ኤልያስ', 'ተራራዎች', 'ከገሀር', 'ደብዳቤዬን', 'መጽሓፍ', 'አስታወሰ', 'የውጤት']
logRegClassificationEvaluator called
logRegClassificationEvaluator called
logRegClassificationEvaluator called
logRegClassificationEvaluator called
logRegClassificationEvaluator called
logRegClassificationEvaluator called
logRegClassificationEvaluator called
logRegClassificationEvaluator called
logRegClassificationEvaluator called
logRegClassificationEvaluator called
dataset keys: dict_keys

In [ ]:
glove_results = evaluation.run(glove_model, output_folder="../MTEB/sparse_results")

##### Clustering

**Task**: k-means model on labeled embeddings for k clusters

**Implementation for TF-IDF**: MTEB only performs clustering on small subset of data (2048 samples) -> should we compute embeddings for this subset or entire data? (This is at least the case for the tasks that get evaluated with AbsTaskClusteringFast.py, maybe not those that get evaluated with AbsTaskClustering.py, but how do I differentiate between these?)

Embedding space is created for the entire corpus, not only the current subset. This is conceptually stronger, because then all clustering happens on the same subspace. Performance-wise, this doesn't change much, speed-wise it does.


**Changes implemented in which files?**
(All of these changes are currently commented out, except for AbsTaskClustering)
- AbsTask.py (this will most likely lead to conflicts!)
- AbsTaskClusteringFast.py
- AbsTaskClustering.py

**Problem Tasks**
- ArxivClusteringP2P

In [ ]:
tasks = mteb.get_tasks(tasks=["StackExchangeClusteringP2P"])
evaluation = mteb.MTEB(tasks=tasks)

TF-IDF on ArXivHierarchicalClusteringS2S

results for computing embeddings on subset: v-measure: 0.4975645682072562

results for computing embeddings on whole data: "v_measure": 0.5006167526797055

TF-IDF on ArxivClusteringS2S

results for computing embeddings on whole data: "v_measure": 0.11543345526606984

TF-IDF on MedrivClusteringS2S

results for computing embeddings on subset: "v_measure": 0.15541653902737848

results for computing embeddings on whole data: 0.15547118002961202

In [ ]:
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")

##### Pair Classification

**Task**: classify a pair of texts as duplicates or paraphrases (binary choice)

**Implementation for TF-IDF**: probably bad performance because tfidf would only recognize a parashrase pair if the same words are used in both. Embedding should be done on the whole dataset, not on the single pairs.

**Changes implemented in which files?**
No changes necessary, embedding already happens for the entire dataset at once

In [ ]:
tasks = mteb.get_tasks(tasks=["TwitterSemEval2015"])
evaluation = mteb.MTEB(tasks=tasks)

In [ ]:
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")

##### Reranking

**Task**: rank texts according to relevance to query. relevance is computed for each document-query pair. each query has a list of positive/relevant and negative/irrelevant documents

**Implementation for TF-IDF**: embed all texts & all queries together

**Changes implemented in which files?**
- RerankingEvaluator.py

**Problem Tasks**
- SciDocsRR
- MindSmallReranking

In [ ]:
tasks = mteb.get_tasks(tasks=["MindSmallReranking"])
evaluation = mteb.MTEB(tasks=tasks)

In [ ]:
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")

##### Retrieval

**Task**: select texts that are relevant for query

**Implementation for TF-IDF**: embed all texts & all queries together

**Changes implemented in which files?**
- AbsTaskRetrieval.py


In [ ]:
tasks = mteb.get_tasks(tasks=["SCIDOCS"])
evaluation = mteb.MTEB(tasks=tasks)

In [ ]:
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")

##### STS (Semantic Textual Similarity)

**Task**: determine similarity between sentence pairs, with ground truth similarities given

**Implementation for TF-IDF**: embed all sentence pairs together or each pair on its own? Maybe test both?

**Changes implemented in which files?**
- STSEvaluator.py

In [ ]:
tasks = mteb.get_tasks(tasks=["STSBenchmark"])
evaluation = mteb.MTEB(tasks=tasks)

In [ ]:
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")

##### Summarization

**Task**: Machine-generated summaries are provided with human-generated quality scores. Task is to predict the summary quality by embedding the machine-generated summaries and corresponding human-generated summaries, and for each summary use the minimal distance to one of the human-generated summaries as a score.

**Implementation for TF-IDF**: 

**Changes implemented in which files?**


In [ ]:
tasks = mteb.get_tasks(tasks=["SummEvalSummarization.v2"])
evaluation = mteb.MTEB(tasks=tasks)

In [ ]:
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")